## Input X
X = Message
Message contains the actual email/SMS text.

## Target Y
Y = Category
Category contains:
ham
spam

## Class label encoding
ham  → 0
spam → 1

## Problem type
Binary Classification

## Main algorithm
This project will compare:
Multinomial Naive Bayes
Logistic Regression
Linear SVM
Random Forest

## The main pipeline will be:
Message
   ↓
Text Cleaning
   ↓
TF-IDF
   ↓
Feature Selection
   ↓
Machine Learning
   ↓
Spam / Ham



In [17]:
## output

from pathlib import Path

# Project root
BASE_DIR = Path.cwd().parent.parent

# Dataset
DATA_DIR = BASE_DIR / "data"

# Outputs
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
RESULT_DIR = OUTPUT_DIR / "results"

# Models
MODEL_DIR = BASE_DIR / "models"

# Create folders
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "email.csv"

print("PROJECT:", BASE_DIR)
print("DATASET:", DATA_PATH)
print("FIGURES:", FIGURE_DIR)
print("RESULTS:", RESULT_DIR)
print("MODELS:", MODEL_DIR)

PROJECT: c:\Users\HP\OneDrive\Desktop\Spam-Email-Detection
DATASET: c:\Users\HP\OneDrive\Desktop\Spam-Email-Detection\data\email.csv
FIGURES: c:\Users\HP\OneDrive\Desktop\Spam-Email-Detection\outputs\figures
RESULTS: c:\Users\HP\OneDrive\Desktop\Spam-Email-Detection\outputs\results
MODELS: c:\Users\HP\OneDrive\Desktop\Spam-Email-Detection\models


# Spam Email Detection

## 01. Dataset Exploration

### Project Objective

The objective of this project is to classify messages as either Spam or Ham using Machine Learning and Natural Language Processing (NLP).

### Dataset

The dataset contains two main columns:

- `Message` → Input Feature (X)
- `Category` → Target / Class Label (Y)

### X and Y

- X = Message
- Y = Category

### Class Labels

- ham = 0
- spam = 1

### Problem Type

Binary Classification

### Main Machine Learning Approach

Text preprocessing → TF-IDF feature extraction → feature selection → classification → evaluation → deployment.

## Import libraries

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


## Set paths

In [19]:
# ============================================================
# PROJECT PATHS
# ============================================================

BASE_DIR = Path.cwd().parent.parent

DATA_DIR = BASE_DIR / "data"

OUTPUT_DIR = BASE_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
RESULT_DIR = OUTPUT_DIR / "results"

MODEL_DIR = BASE_DIR / "models"

# Create folders
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "email.csv"

print("Project folder:", BASE_DIR)
print("Dataset path:", DATA_PATH)
print("Figures path:", FIGURE_DIR)
print("Results path:", RESULT_DIR)

Project folder: c:\Users\HP\OneDrive\Desktop\Spam-Email-Detection
Dataset path: c:\Users\HP\OneDrive\Desktop\Spam-Email-Detection\data\email.csv
Figures path: c:\Users\HP\OneDrive\Desktop\Spam-Email-Detection\outputs\figures
Results path: c:\Users\HP\OneDrive\Desktop\Spam-Email-Detection\outputs\results


## Load dataset

In [20]:
# ============================================================
# LOAD DATASET
# ============================================================

df_raw = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Raw dataset shape:", df_raw.shape)

df_raw.head()

Dataset loaded successfully!
Raw dataset shape: (5573, 2)


,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Save raw dataset information

In [21]:
# ============================================================
# RAW DATASET INFORMATION
# ============================================================

raw_info = pd.DataFrame({
    "Property": [
        "Number of Rows",
        "Number of Columns"
    ],
    "Value": [
        df_raw.shape[0],
        df_raw.shape[1]
    ]
})

raw_info.to_csv(
    RESULT_DIR / "01_raw_dataset_information.csv",
    index=False
)

raw_info

,Property,Value
0,Number of Rows,5573
1,Number of Columns,2


In [22]:
# ============================================================
# DATA CLEANING
# ============================================================

df = df_raw.copy()

# Remove spaces from column names
df.columns = df.columns.str.strip()

# Standardize category values
df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Keep only valid classes
valid_classes = ["ham", "spam"]

df = df[
    df["Category"].isin(valid_classes)
].copy()

# Remove missing values
df = df.dropna(
    subset=["Category", "Message"]
)

# Convert message to string
df["Message"] = df["Message"].astype(str)

print("Before cleaning:", df_raw.shape)
print("After cleaning:", df.shape)

Before cleaning: (5573, 2)
After cleaning: (5572, 2)


In [23]:
# ============================================================
# MISSING VALUE ANALYSIS
# ============================================================

missing_analysis = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": df.isnull().mean() * 100
})

missing_analysis.to_csv(
    RESULT_DIR / "02_missing_value_analysis.csv",
    index=False
)

missing_analysis

,Column,Missing_Count,Missing_Percentage
Category,Category,0,0.0
Message,Message,0,0.0


In [24]:
# ============================================================
# DUPLICATE ANALYSIS
# ============================================================

duplicate_count = df.duplicated().sum()

duplicate_result = pd.DataFrame({
    "Metric": [
        "Duplicate Rows"
    ],
    "Count": [
        duplicate_count
    ]
})

duplicate_result.to_csv(
    RESULT_DIR / "03_duplicate_analysis.csv",
    index=False
)

duplicate_result

,Metric,Count
0,Duplicate Rows,415


In [25]:
# Remove duplicate records

before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)

print("Rows before duplicate removal:", before)
print("Rows after duplicate removal:", after)
print("Duplicates removed:", before - after)

Rows before duplicate removal: 5572
Rows after duplicate removal: 5157
Duplicates removed: 415


In [26]:
# ============================================================
# DEFINE X AND Y
# ============================================================

X = df["Message"]

y = df["Category"]

print("X = Message")
print("Y = Category")

print("\nX shape:", X.shape)
print("Y shape:", y.shape)

print("\nFirst X values:")
print(X.head())

print("\nFirst Y values:")
print(y.head())

X = Message
Y = Category

X shape: (5157,)
Y shape: (5157,)

First X values:
0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: Message, dtype: str

First Y values:
0     ham
1     ham
2    spam
3     ham
4     ham
Name: Category, dtype: str


In [27]:
# ============================================================
# CLASS LABEL ENCODING
# ============================================================

df["label"] = df["Category"].map({
    "ham": 0,
    "spam": 1
})

y_encoded = df["label"]

print("Class Label Encoding")
print("--------------------")
print("ham  = 0")
print("spam = 1")

print("\nClass counts:")
print(df["Category"].value_counts())

Class Label Encoding
--------------------
ham  = 0
spam = 1

Class counts:
Category
ham     4516
spam     641
Name: count, dtype: int64


In [28]:
class_encoding = pd.DataFrame({
    "Original_Class": ["ham", "spam"],
    "Encoded_Label": [0, 1]
})

class_encoding.to_csv(
    RESULT_DIR / "04_class_label_encoding.csv",
    index=False
)

class_encoding

,Original_Class,Encoded_Label
0,ham,0
1,spam,1


In [29]:
# ============================================================
# CLASS DISTRIBUTION
# ============================================================

class_distribution = (
    df["Category"]
    .value_counts()
    .reset_index()
)

class_distribution.columns = [
    "Class",
    "Count"
]

class_distribution["Percentage"] = (
    class_distribution["Count"]
    / class_distribution["Count"].sum()
    * 100
)

class_distribution.to_csv(
    RESULT_DIR / "05_class_distribution.csv",
    index=False
)

class_distribution

,Class,Count,Percentage
0,ham,4516,87.570293
1,spam,641,12.429707


In [30]:
# ============================================================
# FINAL DATASET INFORMATION
# ============================================================

dataset_summary = pd.DataFrame({
    "Metric": [
        "Raw Records",
        "Final Records",
        "Input Feature",
        "Target Variable",
        "Number of Classes",
        "Ham Count",
        "Spam Count",
        "Duplicate Rows Remaining"
    ],
    "Value": [
        len(df_raw),
        len(df),
        "Message",
        "Category",
        2,
        (df["Category"] == "ham").sum(),
        (df["Category"] == "spam").sum(),
        df.duplicated().sum()
    ]
})

dataset_summary.to_csv(
    RESULT_DIR / "06_dataset_summary.csv",
    index=False
)

dataset_summary

,Metric,Value
0,Raw Records,5573
1,Final Records,5157
2,Input Feature,Message
3,Target Variable,Category
4,Number of Classes,2
5,Ham Count,4516
6,Spam Count,641
7,Duplicate Rows Remaining,0


In [31]:
# ============================================================
# CO MAPPING
# ============================================================

co_mapping = pd.DataFrame({
    "CO": [
        "CO1",
        "CO2",
        "CO3",
        "CO4",
        "CO5",
        "CO6"
    ],
    "Implementation": [
        "Dataset → preprocessing → features → training → evaluation → deployment",
        "Supervised learning using Naive Bayes, Logistic Regression and Linear SVM",
        "Random Forest and feature engineering",
        "Unsupervised learning using K-Means and DBSCAN",
        "Cross-validation and classification evaluation metrics",
        "Model packaging and deployment-ready prediction"
    ]
})

co_mapping.to_csv(
    RESULT_DIR / "07_CO_mapping.csv",
    index=False
)

co_mapping

,CO,Implementation
0,CO1,Dataset → preprocessing → features → training ...
1,CO2,"Supervised learning using Naive Bayes, Logisti..."
2,CO3,Random Forest and feature engineering
3,CO4,Unsupervised learning using K-Means and DBSCAN
4,CO5,Cross-validation and classification evaluation...
5,CO6,Model packaging and deployment-ready prediction
